<a href="https://colab.research.google.com/github/RayanMohammed/de-pipeline/blob/main/SyntheaGeneratorNotebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get install -y openjdk-17-jdk-headless
!java -version

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  openjdk-17-jre-headless
Suggested packages:
  openjdk-17-demo openjdk-17-source libnss-mdns fonts-dejavu-extra
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  openjdk-17-jdk-headless openjdk-17-jre-headless
0 upgraded, 2 newly installed, 0 to remove and 1 not upgraded.
Need to get 120 MB of archives.
After this operation, 272 MB of additional disk space will be used.
Ign:1 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 openjdk-17-jre-headless amd64 17.0.19+10-1~22.04.2
Ign:2 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 openjdk-17-jdk-headless amd64 17.0.19+10-1~22.04.2
Err:1 http://security.ubuntu.com/ubuntu jammy-updates/universe amd64 openjdk-17-jre-headless amd64 17.0.19+10-1~22.04.2
  404  Not Found [IP: 91

In [2]:
!wget -q https://github.com/synthetichealth/synthea/releases/download/master-branch-latest/synthea-with-dependencies.jar
!ls -lh synthea-with-dependencies.jar

-rw-r--r-- 1 root root 188M Aug 18 16:19 synthea-with-dependencies.jar


In [3]:
!java -jar synthea-with-dependencies.jar -p 100 --exporter.hospital.fhir.export=false --exporter.practitioner.fhir.export=false
!ls output/fhir | wc -l

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
SLF4J: No SLF4J providers were found.
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#noProviders for further details.
Scanned 90 modules and 157 submodules.
Loading submodule modules/allergies/allergy_panel.json
Loading submodule modules/allergies/drug_allergy_incidence.json
Loading submodule modules/allergies/environmental_allergy_incidence.json
Loading submodule modules/allergies/food_allergy_incidence.json
Loading submodule modules/allergies/immunotherapy.json
Loading submodule modules/allergies/outgrow_env_allergies.json
Loading submodule modules/allergies/outgrow_food_allergies.json
Loading submodule 

In [4]:
!rm -rf output
!java -jar synthea-with-dependencies.jar -p 10000 --exporter.hospital.fhir.export=false --exporter.practitioner.fhir.export=false

Streaming output truncated to the last 5000 lines.
5688 -- Suzi977 Cristen212 Veum823 (58 y/o F) Braintree, Massachusetts  (82636)
5689 -- Sharron285 Schumm995 (55 y/o F) Marshfield, Massachusetts  (77292)
5690 -- Reatha769 Farrell962 (21 y/o F) Amherst Center, Massachusetts  (28925)
5691 -- Guillermo498 Lozada73 (32 y/o M) Wilmington, Massachusetts  (43271)
5692 -- Gisele901 Berge125 (50 y/o F) Lynn, Massachusetts  (70618)
5693 -- Maudie129 Langworth352 (4 y/o F) Boston, Massachusetts  (6912)
5694 -- Wayne846 Bergnaum523 (21 y/o M) Boston, Massachusetts  (28746)
5696 -- Tyrone215 Considine820 (7 y/o M) Northbridge, Massachusetts  (9605)
5695 -- Luciana251 Hauck852 (20 y/o F) Hingham, Massachusetts  (27852)
5698 -- Mindi87 Bartell116 (17 y/o F) Billerica, Massachusetts  (23176)
5697 -- Lou594 Tomika243 Bradtke547 (38 y/o F) Georgetown, Massachusetts  (52137)
5699 -- Marisha663 McKenzie376 (4 y/o F) Lowell, Massachusetts  (6888)
5701 -- Ardella559 Meryl421 Jacobi462 (50 y/o F) New Bedfo

In [5]:
!du -sh output/fhir

45G	output/fhir


In [6]:
%cd output/fhir
!ls *.json | shuf -n 2000 > /content/subset_files.txt
!tar -czf /content/synthea_subset.tar.gz -T /content/subset_files.txt
%cd /content
!ls -lh synthea_subset.tar.gz

/content/output/fhir
/content
-rw-r--r-- 1 root root 597M Sep  8 06:41 synthea_subset.tar.gz


In [10]:
!pip install boto3
import boto3
from google.colab import userdata

s3 = boto3.client(
    "s3",
    endpoint_url=userdata.get("R2_ENDPOINT_URL"),
    aws_access_key_id=userdata.get("R2_ACCESS_KEY_ID"),
    aws_secret_access_key=userdata.get("R2_SECRET_ACCESS_KEY"),
    region_name="auto",
)

In [11]:
s3.upload_file("/content/synthea_subset.tar.gz", "clinical-data-lake", "synthea_subset.tar.gz")